# Customer Churn -- Analysis & EDA

This notebook is the exploratory / reporting layer. All reusable logic (data loading, feature engineering, model training, survival analysis) lives in the `src/` package and is imported here rather than duplicated.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd()))
from src import classifiers, data, explain, features, survival


## Data loading

In [ ]:
df = data.load_data()
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df[df.duplicated()]

## EDA

In [ ]:
df.describe()

In [ ]:
corr = df.drop(columns=['Surname', 'Geography', 'Gender']).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature correlation')
plt.show()

Balance, Age, IsActiveMember and NumOfProducts show the strongest correlation with churn (`Exited`).

## Feature engineering

In [ ]:
df_encoded = features.encode_features(df)
df_encoded.head()

In [ ]:
X, y = features.split_features_target(df_encoded)
X_train, X_test, y_train, y_test = features.train_test_split_data(X, y)
X_train_scaled, X_test_scaled, scaler = features.scale_features(X_train, X_test)

## Modelling

### Random Forest

In [ ]:
rf_model = classifiers.train_random_forest(X_train_scaled, y_train)
rf_result = classifiers.evaluate('RandomForest', y_test, rf_model.predict(X_test_scaled))
print(rf_result.confusion_matrix)
print(rf_result.classification_report)
print(f'Accuracy: {rf_result.accuracy:.4f}')

#### Feature importance

In [ ]:
imp = classifiers.feature_importance(rf_model, features.FEATURES)
plt.figure(figsize=(10, 6))
sns.barplot(data=imp, x='Importance', y='Feature')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

#### SHAP explainability

In [ ]:
shap_values = explain.tree_shap_values(rf_model, X_test_scaled)
shap_values

### Logistic Regression

In [ ]:
logr_model = classifiers.train_logistic_regression(X_train_scaled, y_train)
logr_result = classifiers.evaluate('LogisticRegression', y_test, logr_model.predict(X_test_scaled))
print(logr_result.confusion_matrix)
print(logr_result.classification_report)
print(f'Accuracy: {logr_result.accuracy:.4f}')

### SVM

In [ ]:
svm_model = classifiers.train_svm(X_train_scaled, y_train)
svm_result = classifiers.evaluate('SVM', y_test, svm_model.predict(X_test_scaled))
print(svm_result.confusion_matrix)
print(svm_result.classification_report)
print(f'Accuracy: {svm_result.accuracy:.4f}')

### KNN

In [ ]:
knn_model = classifiers.train_knn(X_train_scaled, y_train)
knn_result = classifiers.evaluate('KNN', y_test, knn_model.predict(X_test_scaled))
print(knn_result.confusion_matrix)
print(knn_result.classification_report)
print(f'Accuracy: {knn_result.accuracy:.4f}')

### Gradient Boosting

In [ ]:
gbm_model = classifiers.train_gbm(X_train_scaled, y_train)
gbm_result = classifiers.evaluate('GBM', y_test, gbm_model.predict(X_test_scaled))
print(gbm_result.confusion_matrix)
print(gbm_result.classification_report)
print(f'Accuracy: {gbm_result.accuracy:.4f}')

### XGBoost

Requires `xgboost` (see `requirements.txt`).

In [ ]:
xgb_model, xgb_best_params = classifiers.train_xgboost(X_train_scaled, y_train)
xgb_result = classifiers.evaluate('XGBoost', y_test, xgb_model.predict(X_test_scaled))
print('Best params:', xgb_best_params)
print(xgb_result.classification_report)
print(f'Accuracy: {xgb_result.accuracy:.4f}')

In [ ]:
imp = classifiers.feature_importance(xgb_model, features.FEATURES)
plt.figure(figsize=(10, 6))
sns.barplot(data=imp, x='Importance', y='Feature')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

### CatBoost

Requires `catboost` (see `requirements-extra.txt`).

In [ ]:
cat_model, cat_best_params = classifiers.train_catboost(X_train_scaled, y_train)
cat_result = classifiers.evaluate('CatBoost', y_test, cat_model.predict(X_test_scaled))
print('Best params:', cat_best_params)
print(cat_result.classification_report)
print(f'Accuracy: {cat_result.accuracy:.4f}')

### LightGBM

Requires `lightgbm` (see `requirements-extra.txt`).

In [ ]:
lgb_model, lgb_best_params = classifiers.train_lightgbm(X_train_scaled, y_train)
lgb_result = classifiers.evaluate('LightGBM', y_test, lgb_model.predict(X_test_scaled))
print('Best params:', lgb_best_params)
print(lgb_result.classification_report)
print(f'Accuracy: {lgb_result.accuracy:.4f}')

### Model comparison

In [ ]:
summary = pd.DataFrame(
    [
        {'Model': r.model_name, 'Accuracy': r.accuracy}
        for r in [rf_result, logr_result, svm_result, knn_result, gbm_result, xgb_result, cat_result, lgb_result]
    ]
).sort_values('Accuracy', ascending=False).reset_index(drop=True)
summary

## Survival analysis

`duration_col` (`Tenure`) must stay in real, unscaled units, so the Cox model is fit on the encoded-but-unscaled features, not `X_train_scaled`.

### Cox Proportional Hazards

In [ ]:
cph = survival.fit_cox_model(X, y)
cph.print_summary()

In [ ]:
sample = X.join(y).iloc[:5]
surv_funcs = cph.predict_survival_function(sample.drop(columns=['Exited']))
surv_funcs.plot()
plt.title('Survival Function for First 5 Customers')
plt.xlabel('Tenure (years)')
plt.ylabel('Survival Probability (1 - churn risk)')
plt.grid(True)
plt.show()